# Optimización de Modelos Conjunto Soleado por GMM

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('/content/drive/MyDrive/Tesina/03_Clusterizacion_CTNET.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,77,0,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,82,0,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,85,0,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,87,0,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,88,0,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,86,0,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,89,0,6,Soleado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,95,0,7,Soleado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,100,0,8,Soleado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,100,1,9,Soleado,Lluvioso


In [3]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [4]:
datos_dia = datos[datos["Cluster GMM"] == "Nublado"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
36,2022-09-02 12:00:00,27523.885172,21,57,4,12,Nublado,Nublado,17036.043251,29196.986647
37,2022-09-02 13:00:00,20596.278869,23,45,5,13,Nublado,Nublado,27523.885172,25478.471342
38,2022-09-02 14:00:00,28500.000000,24,37,6,14,Nublado,Nublado,20596.278869,29057.585772
39,2022-09-02 15:00:00,24647.568577,26,33,5,15,Nublado,Nublado,28500.000000,30000.000000
40,2022-09-02 16:00:00,25500.000000,27,34,4,16,Nublado,Nublado,24647.568577,28062.328964
41,2022-09-02 17:00:00,24281.956494,28,36,2,17,Nublado,Nublado,25500.000000,28786.629243
42,2022-09-02 18:00:00,22733.515002,26,39,1,18,Nublado,Nublado,24281.956494,29900.303971
43,2022-09-02 19:00:00,11972.590689,25,44,1,19,Nublado,Nublado,22733.515002,18282.505369
61,2022-09-03 13:00:00,20400.000000,23,48,5,13,Nublado,Nublado,17723.695569,20596.278869
65,2022-09-03 17:00:00,25602.778606,25,45,5,17,Nublado,Nublado,23122.803757,24281.956494


In [5]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [6]:
X = datos_dia[columns]
X

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
36,21,57,4,12,17036.043251,29196.986647
37,23,45,5,13,27523.885172,25478.471342
38,24,37,6,14,20596.278869,29057.585772
39,26,33,5,15,28500.000000,30000.000000
40,27,34,4,16,24647.568577,28062.328964
...,...,...,...,...,...,...
18281,26,31,4,16,25562.000000,25385.000000
18282,26,32,2,17,25386.000000,22664.000000
18283,25,33,1,18,22872.000000,15736.000000
18284,23,38,0,19,15825.000000,1407.000000


In [7]:
y = datos_dia[['Generación']]
y

,Generación
36,27523.885172
37,20596.278869
38,28500.000000
39,24647.568577
40,25500.000000
...,...
18281,25386.000000
18282,22872.000000
18283,15825.000000
18284,1450.000000


Dividimos entrenamiento, validación y prueba

In [8]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [9]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 3252, y_train: 3252
X_val: 697, y_val: 697
X_test: 697, y_test: 697


## Escalar con MinMaxScaler

In [10]:
from sklearn.preprocessing import MinMaxScaler

In [11]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [12]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.43333333 0.98113208 0.4        0.4        0.56786811 0.97323289]
 [0.5        0.75471698 0.5        0.46666667 0.91746284 0.84928238]
 [0.53333333 0.60377358 0.6        0.53333333 0.68654263 0.96858619]
 ...
 [0.4        0.67924528 0.         1.         0.         0.        ]
 [0.26666667 0.81132075 0.4        0.33333333 0.6396     0.67536667]
 [0.4        0.56603774 0.5        0.4        0.66776667 0.65666667]]
(3252, 6)


In [13]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
36,0.433333,0.981132,0.4,0.400000,0.567868,0.973233
37,0.500000,0.754717,0.5,0.466667,0.917463,0.849282
38,0.533333,0.603774,0.6,0.533333,0.686543,0.968586
39,0.600000,0.528302,0.5,0.600000,0.950000,1.000000
40,0.633333,0.547170,0.4,0.666667,0.821586,0.935411
...,...,...,...,...,...,...
12596,0.633333,0.226415,0.0,0.866667,0.319667,0.173900
12597,0.500000,0.396226,0.0,0.933333,0.026400,0.000000
12598,0.400000,0.679245,0.0,1.000000,0.000000,0.000000
12612,0.266667,0.811321,0.4,0.333333,0.639600,0.675367


In [14]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.5        0.39622642 0.6        0.46666667 0.65776667 0.71383333]
 [0.56666667 0.30188679 0.5        0.53333333 0.63403333 0.62716667]
 [0.63333333 0.22641509 0.4        0.6        0.63616667 0.62633333]
 ...
 [0.4        0.49056604 0.3        0.2        0.37583333 0.7706    ]
 [0.53333333 0.30188679 0.5        0.26666667 0.74553333 0.859     ]
 [0.9        0.05660377 0.7        0.6        0.84236667 0.94003333]]
(697, 6)


In [15]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
12614,0.500000,0.396226,0.6,0.466667,0.657767,0.713833
12615,0.566667,0.301887,0.5,0.533333,0.634033,0.627167
12616,0.633333,0.226415,0.4,0.600000,0.636167,0.626333
12617,0.666667,0.150943,0.3,0.666667,0.655400,0.626067
12618,0.733333,0.113208,0.2,0.733333,0.737567,0.527167
...,...,...,...,...,...,...
14590,0.633333,0.490566,0.0,1.000000,0.009467,0.000000
14601,0.300000,0.716981,0.2,0.133333,0.044967,0.422800
14602,0.400000,0.490566,0.3,0.200000,0.375833,0.770600
14603,0.533333,0.301887,0.5,0.266667,0.745533,0.859000


In [16]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.93333333 0.03773585 0.2        0.66666667 0.83383333 0.91813333]
 [0.96666667 0.03773585 0.2        0.73333333 0.72056667 0.88203333]
 [0.93333333 0.03773585 0.1        0.8        0.67996667 0.7186    ]
 ...
 [0.56666667 0.52830189 0.1        0.8        0.7624     0.52453333]
 [0.5        0.62264151 0.         0.86666667 0.5275     0.0469    ]
 [0.46666667 0.75471698 0.         0.93333333 0.04833333 0.        ]]
(697, 6)


In [17]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
14609,0.933333,0.037736,0.2,0.666667,0.833833,0.918133
14610,0.966667,0.037736,0.2,0.733333,0.720567,0.882033
14611,0.933333,0.037736,0.1,0.800000,0.679967,0.718600
14612,0.900000,0.037736,0.0,0.866667,0.574900,0.273700
14613,0.800000,0.075472,0.0,0.933333,0.218967,0.009467
...,...,...,...,...,...,...
18281,0.600000,0.490566,0.4,0.666667,0.852067,0.846167
18282,0.600000,0.509434,0.2,0.733333,0.846200,0.755467
18283,0.566667,0.528302,0.1,0.800000,0.762400,0.524533
18284,0.500000,0.622642,0.0,0.866667,0.527500,0.046900


In [18]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [19]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.41935484 0.94736842 0.4        0.4        0.56786811 0.97323289]
 [0.48387097 0.73684211 0.5        0.46666667 0.91746284 0.84928238]
 [0.51612903 0.59649123 0.6        0.53333333 0.68654263 0.96858619]
 ...
 [0.5483871  0.52631579 0.1        0.8        0.7624     0.52453333]
 [0.48387097 0.61403509 0.         0.86666667 0.5275     0.0469    ]
 [0.4516129  0.73684211 0.         0.93333333 0.04833333 0.        ]]
(4646, 6)


In [20]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
36,0.419355,0.947368,0.4,0.400000,0.567868,0.973233
37,0.483871,0.736842,0.5,0.466667,0.917463,0.849282
38,0.516129,0.596491,0.6,0.533333,0.686543,0.968586
39,0.580645,0.526316,0.5,0.600000,0.950000,1.000000
40,0.612903,0.543860,0.4,0.666667,0.821586,0.935411
...,...,...,...,...,...,...
18281,0.580645,0.491228,0.4,0.666667,0.852067,0.846167
18282,0.580645,0.508772,0.2,0.733333,0.846200,0.755467
18283,0.548387,0.526316,0.1,0.800000,0.762400,0.524533
18284,0.483871,0.614035,0.0,0.866667,0.527500,0.046900


In [21]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [22]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.91746284]
 [0.68654263]
 [0.95      ]
 ...
 [0.        ]
 [0.66776667]
 [0.65776667]]
(3252, 1)


In [23]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
36,0.917463
37,0.686543
38,0.950000
39,0.821586
40,0.850000
...,...
12596,0.026400
12597,0.000000
12598,0.000000
12612,0.667767


In [24]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[6.34033333e-01]
 [6.36166667e-01]
 [6.55400000e-01]
 [7.37566667e-01]
 [5.70933333e-01]
 [3.33533333e-01]
 [2.61000000e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [6.70400000e-01]
 [6.63633333e-01]
 [6.37766667e-01]
 [6.27166667e-01]
 [6.26333333e-01]
 [6.40166667e-01]
 [6.02466667e-01]
 [3.65333333e-01]
 [3.65333333e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [8.34700000e-01]
 [8.30300000e-01]
 [7.99833333e-01]
 [7.87100000e-01]
 [7.82933333e-01]
 [8.26866667e-01]
 [7.83933333e-01]
 [4.66100000e-01]
 [4.78000000e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [8.34700000e-01]
 [8.40866667e-01]
 [8.04600000e-01]
 [7.83966667e-01]
 [7.82933333e-01]
 [8.25166667e-01]
 [7.76633333e-01]
 [4.66033333e-01]
 [6.56666667e-01]
 [6.38400000e-01]
 [6.27833333e-01]
 [6.29333333e-01]
 [6.26066667e-01]
 [6.02466667e-01]
 [3.65333333e-01]
 [3.47666667e-02]
 [6.34033333e-01]
 [6.27166667e-01]
 [6.33666667e-01]
 [6.26066667e-01]
 [6.02466667e-01]
 [3.65333333e-01]
 [3.28833333e-01]
 [3.33833333e-01]
 [3.163000

In [25]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
12614,0.634033
12615,0.636167
12616,0.655400
12617,0.737567
12618,0.570933
...,...
14590,0.000000
14601,0.375833
14602,0.745533
14603,0.742033


In [26]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.72056667]
 [0.67996667]
 [0.5749    ]
 [0.21896667]
 [0.00756667]
 [0.        ]
 [0.81536667]
 [0.90996667]
 [0.80986667]
 [0.76953333]
 [0.5749    ]
 [0.22023333]
 [0.0078    ]
 [0.        ]
 [0.81026667]
 [0.76496667]
 [0.7186    ]
 [0.2851    ]
 [0.0097    ]
 [0.        ]
 [0.80986667]
 [0.8513    ]
 [0.7186    ]
 [0.2862    ]
 [0.01033333]
 [0.        ]
 [0.83926667]
 [0.72      ]
 [0.68456667]
 [0.5749    ]
 [0.2982    ]
 [0.0105    ]
 [0.        ]
 [0.        ]
 [0.05056667]
 [0.39346667]
 [0.6523    ]
 [0.72796667]
 [0.74653333]
 [0.75363333]
 [0.7465    ]
 [0.7461    ]
 [0.7414    ]
 [0.72333333]
 [0.6804    ]
 [0.57563333]
 [0.22806667]
 [0.00756667]
 [0.        ]
 [0.57076667]
 [0.71923333]
 [0.74653333]
 [0.66036667]
 [0.65023333]
 [0.7461    ]
 [0.74116667]
 [0.6299    ]
 [0.59496667]
 [0.58286667]
 [0.22686667]
 [0.00763333]
 [0.        ]
 [0.4228    ]
 [0.6523    ]
 [0.8983    ]
 [0.6337    ]
 [0.51133333]
 [0.64676667]
 [0.24633333]
 [0.00973333]
 [0.        ]
 [0.81

In [27]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
14609,0.720567
14610,0.679967
14611,0.574900
14612,0.218967
14613,0.007567
...,...
18281,0.846200
18282,0.762400
18283,0.527500
18284,0.048333


In [28]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [29]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.91746284]
 [0.68654263]
 [0.95      ]
 ...
 [0.5275    ]
 [0.04833333]
 [0.        ]]
(4646, 1)


In [30]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
36,0.917463
37,0.686543
38,0.950000
39,0.821586
40,0.850000
...,...
18281,0.846200
18282,0.762400
18283,0.527500
18284,0.048333


## Preparación para Redes Neuronales

In [31]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []

    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada

        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [32]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [33]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (3204, 48, 6), y_train: (3204, 1)
X_val: (649, 48, 6), y_val: (649, 1)
X_test: (649, 48, 6), y_test: (649, 1)


## Optuna

In [34]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.8/231.8 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 6.7 MB/s eta 0:00:00


In [35]:
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

### Optimización de LightGBM

In [36]:
import optuna
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# Definir la función de optimización
def objective(trial):
    # Espacio de búsqueda para los hiperparámetros
    num_leaves = trial.suggest_int('num_leaves', 10, 1000)
    subsample = trial.suggest_float('subsample', 0.1, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.1, 1.0)
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 10, 100)

    # Modelo LightGBM con los hiperparámetros sugeridos
    model = LGBMRegressor(
        num_leaves=num_leaves,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_data_in_leaf=min_data_in_leaf,
        random_state=42
    )

    # Entrenamos el modelo con los datos de entrenamiento
    model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Ejecutar la optimización con 50 iteraciones
LightGBM_study = optuna.create_study(direction="minimize")
LightGBM_study.optimize(objective, n_trials=50)

# Obtener los mejores hiperparámetros
best_params = LightGBM_study.best_params
print("Mejores hiperparámetros:", best_params)

[I 2025-03-14 15:12:54,449] A new study created in memory with name: no-name-3f03476b-33c6-42b3-a776-e237ebf80e3a
[I 2025-03-14 15:12:54,611] Trial 0 finished with value: 0.003783536093271137 and parameters: {'num_leaves': 330, 'subsample': 0.37006545246357303, 'colsample_bytree': 0.477989637553906, 'min_data_in_leaf': 52}. Best is trial 0 with value: 0.003783536093271137.


[LightGBM] [Warning] min_data_in_leaf is set=52, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=52
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=52, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=52
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000888 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 620
[LightGBM] [Info] Number of data points in the train set: 3252, number of used features: 6
[LightGBM] [Info] Start training from score 0.515719
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

[I 2025-03-14 15:12:54,731] Trial 1 finished with value: 0.0038497716377749475 and parameters: {'num_leaves': 253, 'subsample': 0.3754725026664192, 'colsample_bytree': 0.6822636566845305, 'min_data_in_leaf': 40}. Best is trial 0 with value: 0.003783536093271137.
[I 2025-03-14 15:12:54,820] Trial 2 finished with value: 0.0036074113184906957 and parameters: {'num_leaves': 984, 'subsample': 0.4594018619072756, 'colsample_bytree': 0.6629039540838805, 'min_data_in_leaf': 70}. Best is trial 2 with value: 0.0036074113184906957.
[I 2025-03-14 15:12:54,864] Trial 3 finished with value: 0.0046011157374755566 and parameters: {'num_leaves': 82, 'subsample': 0.8742152130842845, 'colsample_bytree': 0.3544013449597859, 'min_data_in_leaf': 100}. Best is trial 2 with value: 0.0036074113184906957.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-14 15:12:54,939] Trial 4 finished with value: 0.0035740050955572387 and parameters: {'num_leaves': 799, 'subsample': 0.819336887221671, 'colsample_bytree': 0.645452583570322, 'min_data_in_leaf': 67}. Best is trial 4 with value: 0.0035740050955572387.
[I 2025-03-14 15:12:55,014] Trial 5 finished with value: 0.0036719742326083633 and parameters: {'num_leaves': 609, 'subsample': 0.9076060474976164, 'colsample_bytree': 0.727627179003159, 'min_data_in_leaf': 58}. Best is trial 4 with value: 0.0035740050955572387.


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-14 15:12:59,640] Trial 47 finished with value: 0.0037515433965059934 and parameters: {'num_leaves': 823, 'subsample': 0.8972897453151496, 'colsample_bytree': 0.8344627083336553, 'min_data_in_leaf': 65}. Best is trial 33 with value: 0.0035648600275307203.
[I 2025-03-14 15:12:59,742] Trial 48 finished with value: 0.0037931241874296535 and parameters: {'num_leaves': 666, 'subsample': 0.3438561411576009, 'colsample_bytree': 0.5186881218719719, 'min_data_in_leaf': 71}. Best is trial 33 with value: 0.0035648600275307203.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-14 15:12:59,834] Trial 49 finished with value: 0.0037358939946032063 and parameters: {'num_leaves': 34, 'subsample': 0.4284004242004754, 'colsample_bytree': 0.723741123679151, 'min_data_in_leaf': 45}. Best is trial 33 with value: 0.0035648600275307203.


[LightGBM] [Warning] min_data_in_leaf is set=45, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=45
Mejores hiperparámetros: {'num_leaves': 990, 'subsample': 0.4913932898599116, 'colsample_bytree': 0.6988987393109299, 'min_data_in_leaf': 71}


### Random Forest

In [37]:
def objective(trial):
    # Definir los hiperparámetros a optimizar
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
    max_depth = trial.suggest_int("max_depth", 10, 50, step=5)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    # Definir el modelo con los hiperparámetros actuales
    RF_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        criterion="squared_error",
        random_state=0
    )

    # Entrenar el modelo
    RF_model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = RF_model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Crear el estudio de optimización
RF_study = optuna.create_study(direction="minimize")
RF_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", RF_study.best_params)

[I 2025-03-14 15:12:59,842] A new study created in memory with name: no-name-bbc9b1da-c759-45f5-ab9d-0e52eea24dd1
[I 2025-03-14 15:13:01,491] Trial 0 finished with value: 0.0035382111655943896 and parameters: {'n_estimators': 200, 'max_depth': 25, 'min_samples_split': 6, 'min_samples_leaf': 7, 'bootstrap': True}. Best is trial 0 with value: 0.0035382111655943896.
[I 2025-03-14 15:13:04,996] Trial 1 finished with value: 0.008150130135145709 and parameters: {'n_estimators': 250, 'max_depth': 25, 'min_samples_split': 14, 'min_samples_leaf': 1, 'bootstrap': False}. Best is trial 0 with value: 0.0035382111655943896.
[I 2025-03-14 15:13:08,014] Trial 2 finished with value: 0.0036430905062935803 and parameters: {'n_estimators': 350, 'max_depth': 40, 'min_samples_split': 12, 'min_samples_leaf': 5, 'bootstrap': True}. Best is trial 0 with value: 0.0035382111655943896.
[I 2025-03-14 15:13:10,081] Trial 3 finished with value: 0.003568261084719641 and parameters: {'n_estimators': 250, 'max_depth':

Mejores hiperparámetros: {'n_estimators': 100, 'max_depth': 30, 'min_samples_split': 16, 'min_samples_leaf': 7, 'bootstrap': True}


### CTNET

In [36]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])

    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

# Construir modelo con hiperparámetros sugeridos
def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout, mlp_dropout):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs

    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)

    for units in mlp_units:
        x = layers.Dense(units, activation="relu")(x)

    x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs)

In [37]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [38]:
# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    head_size = trial.suggest_int("head_size", 2, 8)
    num_heads = trial.suggest_int("num_heads", 2, 8)
    ff_dim = trial.suggest_int("ff_dim", 16, 128, step=16)
    num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 5)
    mlp_units = [trial.suggest_int("mlp_units_1", 64, 512, step=64),
                 trial.suggest_int("mlp_units_2", 32, 256, step=32)]
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Construir el modelo
    model = build_model(
        input_shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]),
        head_size=head_size,
        num_heads=num_heads,
        ff_dim=ff_dim,
        num_transformer_blocks=num_transformer_blocks,
        mlp_units=mlp_units,
        dropout=dropout,
        mlp_dropout=mlp_dropout
    )

    # Compilar y entrenar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Minimizar MSE

# Crear el estudio de optimización
CTNET_study = optuna.create_study(direction="minimize")
CTNET_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", CTNET_study.best_params)


[I 2025-03-14 16:24:33,517] A new study created in memory with name: no-name-6c668fd9-2ad5-4259-b3c6-4f7e36a0d16d
[I 2025-03-14 16:26:03,151] Trial 0 finished with value: 0.08088389039039612 and parameters: {'head_size': 8, 'num_heads': 3, 'ff_dim': 112, 'num_transformer_blocks': 5, 'mlp_units_1': 320, 'mlp_units_2': 32, 'dropout': 0.37550865144773293, 'mlp_dropout': 0.3023382505391271, 'learning_rate': 5.2690503797113496e-05, 'batch_size': 128}. Best is trial 0 with value: 0.08088389039039612.
[I 2025-03-14 16:27:31,463] Trial 1 finished with value: 0.05246976763010025 and parameters: {'head_size': 2, 'num_heads': 4, 'ff_dim': 64, 'num_transformer_blocks': 5, 'mlp_units_1': 192, 'mlp_units_2': 128, 'dropout': 0.2024404970435556, 'mlp_dropout': 0.2708725767838099, 'learning_rate': 6.21478777195467e-05, 'batch_size': 256}. Best is trial 1 with value: 0.05246976763010025.
[I 2025-03-14 16:28:16,373] Trial 2 finished with value: 0.022172562777996063 and parameters: {'head_size': 4, 'num_h

Mejores hiperparámetros: {'head_size': 3, 'num_heads': 5, 'ff_dim': 128, 'num_transformer_blocks': 4, 'mlp_units_1': 64, 'mlp_units_2': 64, 'dropout': 0.24660642380724143, 'mlp_dropout': 0.14852584809648614, 'learning_rate': 0.0007986646130384079, 'batch_size': 256}


### Forescasting

In [39]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Conv1D, BatchNormalization, MaxPooling1D, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters = trial.suggest_categorical("filters", [32, 64, 128])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    lstm_units_1 = trial.suggest_categorical("lstm_units_1", [64, 128, 256])
    lstm_units_2 = trial.suggest_categorical("lstm_units_2", [32, 64, 128])
    lstm_units_3 = trial.suggest_categorical("lstm_units_3", [16, 32, 64])
    dropout_lstm = trial.suggest_float("dropout_lstm", 0.1, 0.5)
    dropout_dense = trial.suggest_float("dropout_dense", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])  # 🔹 Optimización del batch size

    # Construcción del modelo
    model = Sequential()
    model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

    # CNN
    model.add(Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))

    # BiLSTM
    model.add(Bidirectional(LSTM(lstm_units_1, return_sequences=True)))
    model.add(Bidirectional(LSTM(lstm_units_2, return_sequences=True)))
    model.add(Dropout(dropout_lstm))
    model.add(Bidirectional(LSTM(lstm_units_3, return_sequences=False)))

    # Normalización y Dropout
    model.add(BatchNormalization())
    model.add(Dropout(dropout_dense))

    # Capas Densas
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='relu'))

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Entrenar el modelo con validación
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Forecasting_study = optuna.create_study(direction="minimize")
Forecasting_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Forecasting_study.best_params)

[I 2025-03-14 17:13:09,328] A new study created in memory with name: no-name-04f5ea84-658c-4690-b386-a7739f2bb562
[I 2025-03-14 17:14:29,654] Trial 9 finished with value: 0.13768203556537628 and parameters: {'filters': 64, 'kernel_size': 2, 'lstm_units_1': 128, 'lstm_units_2': 64, 'lstm_units_3': 64, 'dropout_lstm': 0.10149417948428048, 'dropout_dense': 0.2538427010548123, 'learning_rate': 0.008672505066956365, 'batch_size': 256}. Best is trial 9 with value: 0.13768203556537628.
[I 2025-03-14 17:14:41,160] Trial 12 finished with value: 0.4284745454788208 and parameters: {'filters': 128, 'kernel_size': 2, 'lstm_units_1': 128, 'lstm_units_2': 64, 'lstm_units_3': 16, 'dropout_lstm': 0.38478086545173995, 'dropout_dense': 0.3055166479540916, 'learning_rate': 0.0016590994889786603, 'batch_size': 128}. Best is trial 9 with value: 0.13768203556537628.
[I 2025-03-14 17:14:52,568] Trial 13 finished with value: 0.3677898943424225 and parameters: {'filters': 32, 'kernel_size': 5, 'lstm_units_1': 6

Mejores hiperparámetros: {'filters': 64, 'kernel_size': 2, 'lstm_units_1': 256, 'lstm_units_2': 64, 'lstm_units_3': 64, 'dropout_lstm': 0.31196916627553595, 'dropout_dense': 0.49436107395722206, 'learning_rate': 0.00881680867870464, 'batch_size': 128}


### Photovoltaic

In [40]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Bidirectional, GRU, MultiHeadAttention
from tensorflow.keras.layers import Flatten, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters_1 = trial.suggest_categorical("filters_1", [32, 64, 128])
    filters_2 = trial.suggest_categorical("filters_2", [64, 128, 256])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    gru_units = trial.suggest_categorical("gru_units", [32, 64, 128])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    dense_units_1 = trial.suggest_categorical("dense_units_1", [32, 64, 128])
    dense_units_2 = trial.suggest_categorical("dense_units_2", [16, 32, 64])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Definir el modelo
    inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

    # Primera capa CNN
    x = Conv1D(filters=filters_1, kernel_size=kernel_size, padding='same', activation='relu')(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    # Segunda capa CNN
    x = Conv1D(filters=filters_2, kernel_size=kernel_size, padding='same', activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Capa BiGRU
    x = Bidirectional(GRU(gru_units, return_sequences=True))(x)

    # Atención MultiHead
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=128)(x, x)

    # Aplanar y Dropout
    x = Flatten()(attention)
    x = Dropout(dropout_rate)(x)
    x = Dense(dense_units_1, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = Dense(dense_units_2, activation="relu")(x)

    # Capa de salida
    outputs = Dense(1, activation="linear")(x)

    # Construcción del modelo
    model = Model(inputs=inputs, outputs=outputs)

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Early Stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

    # Entrenar el modelo
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se detendrá con Early Stopping
        batch_size=batch_size,
        validation_data=(X_val_windowed, y_val_windowed),
        callbacks=[early_stop],
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Photovoltaic_study = optuna.create_study(direction="minimize")
Photovoltaic_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Photovoltaic_study.best_params)

[I 2025-03-14 17:21:29,963] A new study created in memory with name: no-name-febddb7c-399b-4023-a151-185972d7fb78


Epoch 67: early stopping
Restoring model weights from the end of the best epoch: 57.


[I 2025-03-14 17:23:11,946] Trial 8 finished with value: 0.016582762822508812 and parameters: {'filters_1': 64, 'filters_2': 64, 'kernel_size': 2, 'gru_units': 128, 'num_heads': 2, 'dropout_rate': 0.30214995808846173, 'dense_units_1': 128, 'dense_units_2': 64, 'learning_rate': 0.001536761055530737, 'batch_size': 512}. Best is trial 8 with value: 0.016582762822508812.


Epoch 57: early stopping
Restoring model weights from the end of the best epoch: 47.


[I 2025-03-14 17:23:35,047] Trial 10 finished with value: 0.02955016680061817 and parameters: {'filters_1': 32, 'filters_2': 64, 'kernel_size': 5, 'gru_units': 64, 'num_heads': 2, 'dropout_rate': 0.2563503794673836, 'dense_units_1': 128, 'dense_units_2': 32, 'learning_rate': 0.0069657106221652654, 'batch_size': 256}. Best is trial 8 with value: 0.016582762822508812.


...

Restoring model weights from the end of the best epoch: 91.


[I 2025-03-14 17:34:34,068] Trial 40 finished with value: 0.015087977051734924 and parameters: {'filters_1': 128, 'filters_2': 128, 'kernel_size': 2, 'gru_units': 128, 'num_heads': 4, 'dropout_rate': 0.4207136145233795, 'dense_units_1': 64, 'dense_units_2': 64, 'learning_rate': 0.00014469265960224032, 'batch_size': 128}. Best is trial 40 with value: 0.015087977051734924.


Restoring model weights from the end of the best epoch: 91.


[I 2025-03-14 17:34:42,808] Trial 49 finished with value: 0.020243167877197266 and parameters: {'filters_1': 128, 'filters_2': 128, 'kernel_size': 3, 'gru_units': 128, 'num_heads': 4, 'dropout_rate': 0.38840541337378737, 'dense_units_1': 64, 'dense_units_2': 64, 'learning_rate': 0.00018269631293757585, 'batch_size': 256}. Best is trial 40 with value: 0.015087977051734924.


Mejores hiperparámetros: {'filters_1': 128, 'filters_2': 128, 'kernel_size': 2, 'gru_units': 128, 'num_heads': 4, 'dropout_rate': 0.4207136145233795, 'dense_units_1': 64, 'dense_units_2': 64, 'learning_rate': 0.00014469265960224032, 'batch_size': 128}
